# STRC Options Hedge — Three-Leg Comparison

Hedge **STRC** preferred with listed puts on **STRC**, **MSTR**, or **IBIT**. Senior stack for wipeout math: convertible debt principal + STRF market value. Wipeout band: **starts** when seniority-adjusted STRC liquidation hits **par ($100)**; **ends** when it hits **$0**. **Hedge amount** = STRC price minus (strategy.com **USD months of dividend coverage** × STRC monthly coupon). Coverage uses USD reserve ÷ stated preferred coupons (same as the website).

**STRC puts are the direct hedge** — same underlying, same wipeout event. Size to strikes near the **hedge amount** (~$90): a $60 put looks cheap but leaves ~$30 unprotected above the strike; only $85–$95 range is shown. **MSTR** common is junior to prefs — wipeout anchor **$0** (not BTC-scaled); strikes near **hedge amount × MSTR/STRC** (~$112). **mid ÷ days** theta on MSTR (calendar mids on deep OTM are nonsense). **IBIT** uses **vertical put spreads** near the wipeout zone — compare **12/4, 12/5, 10/5** (buy high strike, sell low strike). **$11 is excluded** (not on both expiries; too thin). Theta shows only leg strikes used in those spreads.

Expirations differ by ticker (STRC has a short chain). The three-leg comparison picks the best strike by **max Total Yield ($10k)**; **Hedge Cost ($10k)** is from the same book row (theta + borrow, after tax).

**Prerequisites:** `python fetch_data.py` (fetches market data, treasury, and enriches all option chains with delta/IV)

In [6]:
# --- Edit these ---
COST_OF_CAPITAL = 0.0464
MARGINAL_TAX_RATE = 0.25
CAPITAL = 10_000
STRC_MARGIN_REQUIREMENT = 0.43
STRC_MAX_COVERAGE_GAP = 5.0
MSTR_MAX_COVERAGE_GAP = 8.0

In [7]:
import importlib
from pathlib import Path

import mstr_hedge_helpers
import mstr_liquidation

importlib.reload(mstr_liquidation)
importlib.reload(mstr_hedge_helpers)

from mstr_hedge_helpers import (
    mstr_hedge_reference_strike,
    STRC_PAR_VALUE,
    analyze_hedge_leg,
    comparison_summary,
    liquidation_summary_df,
    load_json,
    ibit_price_strc_zero,
    IBIT_HEDGE_SPREADS,
    senior_claims_usd,
    strc_hedge_amount,
)

mstr_hedge_helpers.COST_OF_CAPITAL = COST_OF_CAPITAL
mstr_hedge_helpers.MARGINAL_TAX_RATE = MARGINAL_TAX_RATE
mstr_hedge_helpers.CAPITAL = CAPITAL
mstr_hedge_helpers.STRC_MARGIN_REQUIREMENT = STRC_MARGIN_REQUIREMENT
mstr_hedge_helpers.STRC_MAX_COVERAGE_GAP = STRC_MAX_COVERAGE_GAP
mstr_hedge_helpers.MSTR_MAX_COVERAGE_GAP = MSTR_MAX_COVERAGE_GAP

OUT = Path('output')

print('Loading treasury + market data...')
treas = load_json(OUT / 'mstr_treasury_extracted_data.json')
mstr_data = load_json(OUT / 'mstr_data.json')
mstr_options = load_json(OUT / 'mstr_options.json')
strc_data = load_json(OUT / 'strc_data.json')
strc_options = load_json(OUT / 'strc_options.json')
ibit_data = load_json(OUT / 'ibit_data.json')
ibit_options = load_json(OUT / 'ibit_options.json')

mstr_spot = mstr_data['current_price']
strc_spot = treas['strc_price']
ibit_spot = ibit_data['current_price']
hedge_amount = strc_hedge_amount(treas)

print('=' * 70)
print('STRC HEDGE BOOK (shared across all legs)')
print('=' * 70)
print(liquidation_summary_df(treas, mstr_spot, ibit_spot).to_string(index=False))
print(f"\nOption chains — MSTR: {mstr_data['num_expirations']}  |  STRC: {strc_data['num_expirations']}  |  IBIT: {ibit_data['num_expirations']}")

Loading treasury + market data...
STRC HEDGE BOOK (shared across all legs)
                             Metric          Value
         USD Reserve (strategy.com) $1,000,000,000
Preferred Annual Dividends — Stated      $1,676.4M
    USD Months of Dividend Coverage            7.2
         Convertible Debt Principal $6,713,750,000
                  STRF Market Value $1,190,239,131
                Total Senior Claims $7,903,989,131
                         STRC Price         $84.86
                STRC Monthly Coupon         $0.958
       Not Hedged (7.2 mo × coupon)          $6.86
                       Hedge Amount         $78.00
  BTC — wipeout starts (STRC = par)        $20,578
     BTC — wipeout ends (STRC = $0)         $8,168
                        Current BTC        $58,579
                       Current MSTR         $86.93
     MSTR — wipeout anchor (common)          $0.00
      MSTR — hedge reference strike         $79.90
              IBIT — wipeout starts         $11.69
       

In [8]:
leg_configs = [
    ('STRC puts', strc_data, strc_options, strc_spot, None),
    ('MSTR puts', mstr_data, mstr_options, mstr_spot, None),
    ('IBIT put spreads', ibit_data, ibit_options, ibit_spot, ibit_price_strc_zero(treas, ibit_spot)),
]

leg_results = []
for name, data, options, spot, wipeout in leg_configs:
    print('\n' + '=' * 70)
    print(f'{name.upper()} — THETA & HEDGE SIZING')
    print('=' * 70)
    leg = analyze_hedge_leg(name, data, options, treas, spot=spot, wipeout_price=wipeout)
    leg_results.append(leg)
    if 'STRC' in name.upper():
        print(f"Spot: ${spot:.2f}  |  Hedge: ${hedge_amount:.2f}  |  "
              f"Cash-covered floor: ${treas['strc_price'] - hedge_amount:.2f}")
    elif 'MSTR' in name.upper():
        ref = leg.get('hedge_reference_strike', mstr_hedge_reference_strike(spot, treas))
        print(f"Spot: ${spot:.2f}  |  MSTR wipeout: $0 (common)  |  "
              f"Hedge reference strike: ${ref:.2f}")
    elif 'IBIT' in name.upper():
        print(f"Spot: ${spot:.2f}  |  Spread payoff @ STRC liquidation $0")
    else:
        print(f"Spot: ${spot:.2f}")
    if 'STRC' in name.upper():
        print(f"Theta expiration: {leg.get('last_expiration')} (mid / days to expiry)  |  "
              f"Hedge amount: ${hedge_amount:.2f}")
    elif 'MSTR' in name.upper():
        print(f"Theta expiration: {leg.get('last_expiration')} (mid / days to expiry)  |  "
              f"±${MSTR_MAX_COVERAGE_GAP:.0f} from reference")
    else:
        print(f"Theta expirations: {leg.get('second_last_expiration')} → {leg.get('last_expiration')}")
    if leg['theta'].empty:
        print('No theta table.')
        continue
    if 'IBIT' in name.upper():
        theta_disp = leg['theta'][['strike', 'mid_price', 'openInterest', 'estimated_theta']].copy()
        theta_disp.columns = ['Strike', 'Mid', 'OI', 'Theta $/day (leg)']
        print('Per-leg theta (calendar when both expiries list strike; else mid/days):')
    elif 'STRC' in name.upper() or 'MSTR' in name.upper():
        theta_disp = leg['theta'][['strike', 'mid_price', 'estimated_theta']].copy()
        if 'openInterest' in leg['theta'].columns:
            theta_disp['openInterest'] = leg['theta']['openInterest'].values
            theta_disp = theta_disp[['strike', 'mid_price', 'openInterest', 'estimated_theta']]
            theta_disp.columns = ['Strike', 'Mid', 'OI', 'Theta $/day']
        else:
            theta_disp.columns = ['Strike', 'Mid', 'Theta $/day']
    elif 'mid_price_last' in leg['theta'].columns:
        theta_disp = leg['theta'][['strike', 'mid_price_last', 'mid_price_second_last', 'mid_price_diff', 'estimated_theta']].copy()
        theta_disp.columns = ['Strike', 'Mid (Last)', 'Mid (2nd Last)', 'Diff', 'Theta $/day']
    else:
        theta_disp = leg['theta'][['strike', 'mid_price', 'days_to_expiration', 'estimated_theta']].copy()
        theta_disp.columns = ['Strike', 'Mid', 'Days', 'Theta $/day']
    print(theta_disp.to_string(index=False))
    if leg['hedge_calc'].empty:
        print('No hedge table.')
        continue
    if 'IBIT' in name.upper():
        pairs = ', '.join(f'{lo:g}/{hi:g}' for lo, hi in IBIT_HEDGE_SPREADS)
        print(f"Candidate spreads: {pairs}  (buy long / sell short)")
        opt = leg['hedge_calc'][
            ['spread', 'mid_price_long', 'mid_price_short', 'mid_price', 'spread_width',
             'payoff_at_wipeout', 'open_interest_long', 'open_interest_short',
             'contracts_needed', 'annualized_cost_after_tax_pct']
        ].copy()
        opt.columns = ['Spread', 'Long Mid', 'Short Mid', 'Net Debit', 'Width',
                       'Payoff@Wipeout', 'OI Long', 'OI Short', 'Spreads Needed', 'After-Tax Cost']
    else:
        cols = ['strike', 'mid_price', 'contracts_needed', 'annualized_cost_of_capital',
                'annualized_theta_cost', 'annualized_cost_pct', 'annualized_cost_after_tax_pct']
        if 'coverage_gap' in leg['hedge_calc'].columns:
            cols = ['strike', 'coverage_gap', 'mid_price', 'contracts_needed',
                    'annualized_cost_after_tax_pct']
            if 'STRC' in name.upper():
                print(f'STRC strikes within ${STRC_MAX_COVERAGE_GAP:.0f} of hedge amount (excludes deep OTM like $60).')
            elif 'MSTR' in name.upper():
                ref = leg.get('hedge_reference_strike', float('nan'))
                print(f'MSTR strikes within ${MSTR_MAX_COVERAGE_GAP:.0f} of reference ${ref:.2f} (wipeout @ $0).')
        opt = leg['hedge_calc'][cols].copy()
        if 'coverage_gap' in opt.columns:
            opt.columns = ['Strike', 'Unhedged Gap', 'Mid', 'Contracts', 'After-Tax Cost']
        else:
            opt.columns = ['Strike', 'Mid', 'Contracts', 'Ann. COC', 'Ann. Theta', 'Ann. Cost', 'After-Tax Cost']
    print(f"\nCost of capital: {COST_OF_CAPITAL*100:.2f}%  |  Tax: {MARGINAL_TAX_RATE*100:.0f}%  |  Hedge: ${hedge_amount:.2f}")
    print(opt.to_string(index=False))


STRC PUTS — THETA & HEDGE SIZING
Spot: $84.86  |  Hedge: $78.00  |  Cash-covered floor: $6.86
Theta expiration: 2026-12-18 (mid / days to expiry)  |  Hedge amount: $78.00
 Strike  Mid  Theta $/day
   75.0 7.35     0.042982
   80.0 8.90     0.052047
STRC strikes within $5 of hedge amount (excludes deep OTM like $60).

Cost of capital: 4.64%  |  Tax: 25%  |  Hedge: $78.00
 Strike  Unhedged Gap  Mid  Contracts  After-Tax Cost
   75.0      3.000211 7.35   1.152996       16.824006
   80.0      0.000000 8.90   1.097049       18.934160

MSTR PUTS — THETA & HEDGE SIZING
Spot: $86.93  |  MSTR wipeout: $0 (common)  |  Hedge reference strike: $79.90
Theta expiration: 2028-12-15 (mid / days to expiry)  |  ±$8 from reference
 Strike   Mid  Theta $/day
   75.0 33.25     0.036986
   80.0 38.45     0.042770
   85.0 40.40     0.044939
MSTR strikes within $8 of reference $79.90 (wipeout @ $0).

Cost of capital: 4.64%  |  Tax: 25%  |  Hedge: $78.00
 Strike  Unhedged Gap   Mid  Contracts  After-Tax Cost


In [9]:
for leg in leg_results:
    print('\n' + '=' * 70)
    print(f"{leg['leg'].upper()} — $10k STRC BOOK")
    print('=' * 70)
    book = leg.get('book')
    if book is None or book.empty:
        print('No book table — enrich options with ibit_option_deltas.py for this leg.')
        continue
    hdr = f"Capital: ${CAPITAL:,.0f}  |  STRC margin: {STRC_MARGIN_REQUIREMENT*100:.0f}%  |  Hedge: ${hedge_amount:.2f}"
    if 'STRC' in leg['leg'].upper():
        hdr += f"  |  Cash-covered floor: ${treas['strc_price'] - hedge_amount:.2f}"
    elif 'MSTR' in leg['leg'].upper():
        hdr += "  |  MSTR wipeout: $0 (common)"
    elif 'IBIT' in leg['leg'].upper():
        hdr += "  |  IBIT spread @ STRC liq $0"
    print(hdr)
    strike_col = 'spread' if 'spread' in book.columns else 'strike'
    display_table = book[[strike_col, 'num_shares', 'total_purchase', 'dividend', 'borrow_cost',
                          'theta_cost', 'tax_benefit', 'total_yield', 'total_cash_flow']].copy()
    display_table['num_shares'] = display_table['num_shares'].round(2)
    display_table['dividend'] = display_table['dividend'].round(2)
    display_table['total_purchase'] = display_table['total_purchase'].round(2)
    display_table['borrow_cost'] = display_table['borrow_cost'].round(2)
    display_table['theta_cost'] = display_table['theta_cost'].round(2)
    display_table['tax_benefit'] = display_table['tax_benefit'].round(2)
    display_table['total_yield'] = display_table['total_yield'].round(2)
    display_table['total_cash_flow'] = display_table['total_cash_flow'].round(2)
    display_table.columns = ['Strike', 'Num shares', 'Total Purchase', 'Dividend', 'Borrow Cost',
                             'Theta Cost', 'Tax Benefit', 'Total Yield', 'Total Cash Flow']
    print(display_table.to_string(index=False))
    print()
    if 'STRC' not in leg['leg'].upper():
        pnl = leg.get('pnl')
        if pnl is not None and not pnl.empty:
            print('\nOption book P&L ($) — parallel shock -50% … +50%')
            print(pnl.round(2).to_string(index=False))
        else:
            print('\nP&L sensitivity skipped: no delta on chain.')


STRC PUTS — $10k STRC BOOK
Capital: $10,000  |  STRC margin: 43%  |  Hedge: $78.00  |  Cash-covered floor: $6.86
 Strike  Num shares  Total Purchase  Dividend  Borrow Cost  Theta Cost  Tax Benefit  Total Yield  Total Cash Flow
   75.0       222.4        20757.46   2557.58      -499.15    -4025.70      1131.21      -836.05         -1967.26
   80.0       216.2        20457.62   2486.30      -485.23    -4508.85      1248.52     -1259.26         -2507.78


MSTR PUTS — $10k STRC BOOK
Capital: $10,000  |  STRC margin: 43%  |  Hedge: $78.00  |  MSTR wipeout: $0 (common)
 Strike  Num shares  Total Purchase  Dividend  Borrow Cost  Theta Cost  Tax Benefit  Total Yield  Total Cash Flow
   75.0      101.41        14905.22   1166.21      -227.60    -2559.42       696.76      -924.05         -1620.81
   80.0       92.02        14451.09   1058.24      -206.53    -2698.61       726.28     -1120.61         -1846.89
   85.0       93.33        14514.47   1073.31      -209.47    -2679.18       722.16    

In [10]:
cmp = comparison_summary(leg_results)
print('=' * 70)
print('THREE-LEG COMPARISON (best $10k book yield per leg; hedge cost from same row)')
print('=' * 70)
print(cmp.to_string(index=False))

THREE-LEG COMPARISON (best $10k book yield per leg; hedge cost from same row)
             Leg             Expirations  Strikes Best Strike  Hedge Cost ($10k)  Total Yield ($10k)
       STRC puts 2026-09-18 → 2026-12-18        2        75.0            3393.63             -836.05
       MSTR puts 2028-06-16 → 2028-12-15        3        75.0            2090.27             -924.05
IBIT put spreads 2028-06-16 → 2028-12-15        3        10/5            1381.69             1074.26
